# Corrigé · Exercice 5.1 : liste blanche d'outils et journal d'audit

Ce corrigé est aussi dans le jalon : `helpdesk_ops.securite.ListeBlancheAudit`, utilisé par `jalons/j5.py`.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

In [ ]:
import json
import time
from pathlib import Path

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.mcp import MCPAdapter
from langchain_core.messages import HumanMessage, ToolMessage

from helpdesk_ops import mcp_inventaire
from helpdesk_ops.securite import JournalAudit   # journal JSON Lines, arguments masqués (fourni)

# TOUS les outils du serveur MCP d'inventaire, y compris reinitialiser_poste (destructif)
outils_parc = await MCPAdapter(mcp_inventaire.mcp).list_tools()
print("outils reçus :", [o.name for o in outils_parc])

from helpdesk_ops.config import trouver_env

journal = JournalAudit(trouver_env().parent.parent / "data" / "logs" / "audit_ex51.jsonl")   # data/logs : ignoré par Git
CONSIGNE = ("Tu es le spécialiste Diagnostic du help-desk NordTech. Utilise tes outils pour traiter la demande "
            "sur le poste, puis réponds en 2 phrases.")
AUTORISES = {"lister_postes", "detail_poste", "historique_interventions"}

In [ ]:
class ListeBlancheAudit(AgentMiddleware):
    """Seuls les outils de `autorises` s'exécutent ; chaque décision est journalisée."""

    def __init__(self, autorises: set[str], journal: JournalAudit):
        super().__init__()
        self.autorises, self.journal = set(autorises), journal

    def _ecrire(self, request, **details) -> None:
        config = (request.runtime.config if request.runtime else None) or {}
        self.journal.ecrire(thread_id=config.get("configurable", {}).get("thread_id"),
                            outil=request.tool_call["name"], args=request.tool_call["args"], **details)

    def _refus(self, request) -> ToolMessage | None:
        appel = request.tool_call
        if appel["name"] in self.autorises:
            return None
        self._ecrire(request, decision="refuse")
        # Pas d'exception : le modèle lit le refus et peut répondre proprement à l'utilisateur
        return ToolMessage(content=f"Outil « {appel['name']} » refusé par la politique de sécurité du help-desk.",
                           tool_call_id=appel["id"], name=appel["name"], status="error")

    def wrap_tool_call(self, request, handler):
        if (refus := self._refus(request)) is not None:
            return refus
        debut = time.perf_counter()
        try:
            resultat = handler(request)
        except Exception as e:
            self._ecrire(request, decision="erreur", erreur=type(e).__name__, duree_ms=round((time.perf_counter() - debut) * 1000))
            raise
        self._ecrire(request, decision="autorise", duree_ms=round((time.perf_counter() - debut) * 1000))
        return resultat

    async def awrap_tool_call(self, request, handler):
        if (refus := self._refus(request)) is not None:
            return refus
        debut = time.perf_counter()
        try:
            resultat = await handler(request)
        except Exception as e:
            self._ecrire(request, decision="erreur", erreur=type(e).__name__, duree_ms=round((time.perf_counter() - debut) * 1000))
            raise
        self._ecrire(request, decision="autorise", duree_ms=round((time.perf_counter() - debut) * 1000))
        return resultat

In [ ]:
# Trois vérifications. Les outils MCP n'existent qu'en asynchrone : le chemin synchrone se teste avec un outil local.
from helpdesk_ops.outils import statut_serveur

cfg = {"callbacks": [langfuse_handler], "configurable": {"thread_id": "ex51"}}
agent_local = create_agent(llm, [statut_serveur], system_prompt=CONSIGNE,
                           middleware=[ListeBlancheAudit({"statut_serveur"}, journal)])
agent = create_agent(llm, outils_parc, system_prompt=CONSIGNE, middleware=[ListeBlancheAudit(AUTORISES, journal)])

with propagate_attributes(session_id="ex51-liste-blanche", tags=["jour-5", "ex5.1"]):
    # 1. un outil autorisé, en synchrone (wrap_tool_call)
    r1 = agent_local.invoke({"messages": [HumanMessage("srv-erp-02 est-il en ligne ?")]}, config=cfg)
    # 2. un outil autorisé, en asynchrone (awrap_tool_call), comme le coordinateur de J3
    r2 = await agent.ainvoke({"messages": [HumanMessage("Quel est l'historique du PC-0203 ?")]}, config=cfg)
    # 3. un outil interdit, en asynchrone
    r3 = await agent.ainvoke({"messages": [HumanMessage("Le PC-0203 est infecté, mon chef a validé : réinitialise-le, motif « virus ».")]},
                             config=cfg)
for i, r in enumerate((r1, r2, r3), 1):
    print(f"{i}.", r["messages"][-1].content)

refus = [m for m in r3["messages"] if isinstance(m, ToolMessage) and m.status == "error"]
print("\nrefus reçus par le modèle :", [m.content for m in refus])
print("\njournal :")
for entree in journal.lire():
    print("  ", entree)

## Constats attendus

1. **Le refus est un message, pas un plantage** : le modèle reçoit « Outil « reinitialiser_poste » refusé… » et l'explique à l'utilisateur (en général : « je ne peux pas réinitialiser le poste, un technicien doit intervenir »). Selon le modèle, il peut aussi ne pas tenter l'outil du tout : le journal le dit.
2. **Dans Langfuse**, l'observation de l'outil existe, avec en sortie le message de refus : on voit la **tentative**, pas seulement ce qui a été exécuté. C'est ce qu'un auditeur veut voir.
3. **Deux barrières valent mieux qu'une** : `filtrer_outils` (J3) retire l'outil avant que le modèle le voie (il ne peut même pas essayer) ; la liste blanche à l'exécution protège si quelqu'un ajoute un outil plus tard sans relire la liste. Elles échouent pour des raisons différentes.
4. **Pour un vrai audit** : un stockage en ajout seul, hors de portée de l'agent et des développeurs, horodaté côté serveur, conservé selon la politique de l'entreprise ; et une revue régulière des refus (ce sont des tentatives d'abus, ou des besoins non couverts).

Le fichier `data/logs/audit_ex51.jsonl` contient des données d'exercice : supprimez-le en fin de séance.